# Structure that sees a reversed ordering

Structure correlates the survey's pairwise nEMD distances with the model's, so it
cannot see direction: a model that reverses the ordering of the cells keeps the
same distances and the same score. This notebook scores three readings of
Structure that can, beside the original, and leaves the original untouched.

Every reading starts from one anchor per cell, its nEMD to the distribution with
all mass on the first answer option, which is the cell's normalized mean response:

$$a_i = \mathrm{nEMD}(p_i, r), \qquad r = (1, 0, \dots, 0).$$

The orientation of a set of cells is the Spearman correlation between the survey
anchors and the model anchors,

$$\rho_{\mathrm{orient}} = \mathrm{Spearman}(a^{H}, a^{L}),$$

and a row is flagged `orientation_reversed` when its Structure is positive and
its orientation negative.

| Variant | Structure |
| --- | --- |
| `original` | $\max(0, \rho)$ |
| `gated` | the original where $\rho_{\mathrm{orient}} > 0$, else 0 |
| `capped` | $\min(S_{\mathrm{struct}}, \max(0, \rho_{\mathrm{orient}}))$ |
| `signed` | $\max(0, \rho_{\mathrm{signed}})$ |

$\rho_{\mathrm{signed}}$ is the same Spearman correlation over pairs of cells with
a sign on the model's distance. Each pair is read from the cell with the lower
survey mean to the one with the higher: the survey distance stays positive, the
model distance is positive when the model orders the pair the same way, negative
when it reverses it, zero when its means tie. A pair whose survey means tie has
no direction and is left out. `signed` leaves out exact ties only;
`signed_tie_01`, `signed_tie_02` and `signed_tie_05` also leave out pairs whose
survey means differ by at most 0.01, 0.02 and 0.05 of the scale, and
`signed_tie_se` those within 1.96 standard errors of the difference, from each
cell's response variance and respondent count.

An undefined orientation or signed correlation earns no Structure, as a model
with one answer for every cell earns none today. Each variant's PFS is the same
geometric mean of accuracy, adaptability and its own Structure.

Reads the runs the manuscript reports, every replicate, and reconciles against
`population_fidelity_groups.csv` and `population_fidelity_paired.csv`. Writes
`population_fidelity_orientation_groups.csv`,
`population_fidelity_orientation_paired.csv`,
`population_fidelity_orientation_summary.csv` and the plates under
`figures/fidelity/orientation/`.

In [1]:
from culture.fidelity import GROUPS_TABLE, PAIRED_TABLE
from culture.fidelity_baseline import manuscript_runs
from culture.fidelity_orientation import (
    ORIENTATION_GROUPS_TABLE,
    ORIENTATION_PAIRED_TABLE,
    ORIENTATION_SUMMARY_TABLE,
    build_orientation,
    check_orientation,
    orientation_plates,
    orientation_summary,
)
from culture.tables import read_csv
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
runs = manuscript_runs(every_replicate=True)
frame, paired = build_orientation(runs=runs)
print(len(frame), "group rows and", len(paired), "paired rows over", frame["source"].nunique(), "runs")

9548 group rows and 5952 paired rows over 57 runs


## Reconciling against the groups table

Every row is scored by `group_fidelity` first, so the original columns are the
original numbers. The check asserts that they equal
`population_fidelity_groups.csv` and `population_fidelity_paired.csv` row for
row, that the cap never scores above the gate nor the gate above the original,
that every variant score stays in the unit interval, and that a wider tie
tolerance never keeps more pairs.

In [3]:
groups, published_paired = read_csv(GROUPS_TABLE), read_csv(PAIRED_TABLE)
assert groups is not None and published_paired is not None
matched = check_orientation(frame, paired, groups, published_paired)
print(matched, "rows carry the published scores unchanged")

15500 rows carry the published scores unchanged


## The tables

`population_fidelity_orientation_groups.csv` holds the groups table with
`rho_orientation`, `orientation_reversed`, the signed correlation and its pair
count under each tie rule, and `score_structure_<variant>` and `pfs_<variant>`
for every variant. `population_fidelity_orientation_paired.csv` holds the same
beside the as-released condition, with the change in each. Both keep every run.

`population_fidelity_orientation_summary.csv` reads the first run, one row per
variant in four scopes: `population`, the pooled combinations; `subgroups`,
every demographic level of the sweep's conditions; `home`, the German variant
inside Germany and the Mexican-Spanish variant inside Mexico; and `levels`,
every demographic level with the Mixtral references kept, the count the
metric-sensitivity table reports.

In [4]:
summary = orientation_summary(frame, paired)
write_csv(frame, ORIENTATION_GROUPS_TABLE)
write_csv(paired, ORIENTATION_PAIRED_TABLE)
write_csv(summary, ORIENTATION_SUMMARY_TABLE)
print(len(summary), "summary rows over", summary["variant"].nunique(), "variants")

32 summary rows over 8 variants


## The plates

The component and change plates of the main report, drawn again with the
`capped` and the `signed` Structure, each variant in its own folder under
`figures/fidelity/orientation/`. The stems are the ones the main report uses.

In [5]:
files = orientation_plates(frame, paired)
print(len(files), "files")

160 files


## Reading the table

The first block reads the 164 pooled combinations of the first run. The gate and
the cap only lower a score. Both set to zero the 19 combinations whose Structure
is positive and whose orientation is negative, which takes the zero scores from
30 to 49, and the cap also lowers 13 whose orientation is positive but below
their Structure. The signed correlation moves 151 of the 164 and raises 105 of
them: the sign carries the ordering of the cells, which agrees with the survey
more often than the distances do, so the signed correlation follows the
orientation (rank correlation 0.97 across the 164) more than the original
Structure (0.55). The tie rule changes little; the standard-error rule keeps 43%
of the pairs on average and moves the counts by a few.

The two blocks below read the mean change in PFS after fine-tuning, on the
pooled cells and on the target country's cells. Every variant makes each of the
four changes more negative than the original does.

In [6]:
pooled = summary[summary["scope"].eq("population")].set_index("variant")
levels = ["n_changed", "n_raised", "n_lowered", "n_zero_pfs", "n_structure_binding", "pfs_median", "rank_vs_original"]
print(pooled[levels].round(3).to_string())
changes = [f"{culture}_{reading}" for culture in ("german", "spanish_mx") for reading in ("delta_pfs_mean", "n_pfs_up", "n_pfs_down")]
for scope in ("population", "home"):
    print(f"\n{scope}")
    print(summary[summary["scope"].eq(scope)].set_index("variant")[changes].round(3).to_string())

               n_changed  n_raised  n_lowered  n_zero_pfs  n_structure_binding  pfs_median  rank_vs_original
variant                                                                                                     
original               0         0          0          30                  159       0.254             1.000
gated                 19         0         19          49                  160       0.240             0.918
capped                32         0         32          49                  160       0.220             0.900
signed               151       105         46          40                  151       0.303             0.654
signed_tie_01        151       104         47          41                  151       0.301             0.652
signed_tie_02        151       104         47          40                  151       0.302             0.650
signed_tie_05        151       103         48          41                  152       0.307             0.645
signed_tie_se      